In [ ]:
Data Loader

In [ ]:
# Delete the "Autre Surface" patches

import os, glob

DATA_DIR = os.path.expanduser('~/Desktop/training_data/')

to_delete = glob.glob(os.path.join(DATA_DIR, '*_Autre surface*.tif'))
print(f"Found {len(to_delete)} 'Autre surface' patches to delete")

for f in to_delete:
    os.remove(f)

print("Done!")

In [5]:
import os
import glob

# Utilisation du chemin absolu pour éviter toute confusion
DATA_DIR = r'C:\Users\clari\Documents\MA3\Design_Project\training_data_extracted'

# On utilise ** pour chercher dans tous les sous-dossiers
# recursive=True est indispensable pour descendre dans l'arborescence
search_pattern = os.path.join(DATA_DIR, '**', '*Autre surface*.tif')

# glob.glob récupère tous les fichiers correspondants
to_delete = glob.glob(search_pattern, recursive=True)

print(f"Recherche dans : {DATA_DIR}")
print(f"Trouvé : {len(to_delete)} patches 'Autre surface' à supprimer.")

# Suppression effective
for f in to_delete:
    try:
        os.remove(f)
        # print(f"Supprimé : {os.path.basename(f)}") # Optionnel : pour voir chaque fichier
    except OSError as e:
        print(f"Erreur lors de la suppression de {f} : {e}")

print("---")
print("Opération terminée !")

Recherche dans : C:\Users\clari\Documents\MA3\Design_Project\training_data_extracted
Trouvé : 10551 patches 'Autre surface' à supprimer.
---
Opération terminée !


Filtrage VARI sur 1000 images 'PLUS'...


100%|██████████| 1000/1000 [00:16<00:00, 59.79it/s]


In [30]:
import os
import shutil
import numpy as np
import random
from PIL import Image
from tqdm import tqdm

def filter_keep_yellow_fields(input_dir, output_dir, rejected_dir, sample_size=1000, white_limit=20):
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    if not os.path.exists(rejected_dir): os.makedirs(rejected_dir)

    all_files = [f for f in os.listdir(input_dir) if f.endswith('.tif')]
    sample_files = random.sample(all_files, min(len(all_files), sample_size))
    
    stats = {"kept": 0, "rejected": 0}

    for img_name in tqdm(sample_files):
        img_path = os.path.join(input_dir, img_name)
        
        # On garde tous les MOINS d'office
        if "MOINS" in img_name.upper():
            shutil.copy(img_path, os.path.join(output_dir, img_name))
            stats["kept"] += 1
            continue

        img = Image.open(img_path).convert('RGB')
        arr = np.array(img).astype(float)
        R, G, B = arr[:,:,0], arr[:,:,1], arr[:,:,2]
        
        # 1. Détection du BLANC SATURÉ (Les grosses routes blanches)
        # Un pixel est "blanc" si R,G,B > 220
        is_white = (R > 220) & (G > 220) & (B > 220)
        white_percent = (np.sum(is_white) / is_white.size) * 100
        
        # 2. Détection du GRIS (Routes goudronnées, toits béton)
        # Dans le gris, R, G et B sont presque égaux. 
        # Dans un champ marron/jaune, R est beaucoup plus grand que B.
        color_diff = np.abs(R - B) 
        is_gray = color_diff < 15 # Si la différence entre rouge et bleu est faible, c'est du gris
        gray_percent = (np.sum(is_gray) / is_gray.size) * 100

        # LOGIQUE DE REJET
        # On ne rejette que si l'image contient TROP de blanc (route) 
        # ou si elle est INTEGRALEMENT grise (parking/toit)
        is_bad = False
        if white_percent > white_limit: # Trop de blanc = Route
            is_bad = True
        if gray_percent > 95: # 95% de gris = Parking/Toit usine
            is_bad = True

        if is_bad:
            shutil.copy(img_path, os.path.join(rejected_dir, img_name))
            stats["rejected"] += 1
        else:
            shutil.copy(img_path, os.path.join(output_dir, img_name))
            stats["kept"] += 1

    print(f"Terminé. Gardés: {stats['kept']}, Rejetés: {stats['rejected']}")

# --- EXECUTION ---
source = r"C:\Users\clari\Documents\MA3\Design_Project\training_data_extracted\training_data"
filter_keep_yellow_fields(source, "dataset_KEEP", "dataset_REJECTED", white_limit=5)

100%|██████████| 1000/1000 [00:11<00:00, 84.26it/s]

Terminé. Gardés: 982, Rejetés: 18


In [33]:
import os
import shutil
import numpy as np
import random
from PIL import Image
from skimage.feature import graycomatrix, graycoprops
from tqdm import tqdm

def filter_by_texture(input_dir, output_dir, rejected_dir, sample_size=1000, homo_thresh=0.75):
    """
    Trie les patches 'PLUS' en identifiant les toits et routes par analyse de texture (GLCM Homogénéité).
    """
    for d in [output_dir, rejected_dir]:
        if not os.path.exists(d): os.makedirs(d)

    all_files = [f for f in os.listdir(input_dir) if f.endswith('.tif')]
    sample_files = random.sample(all_files, min(len(all_files), sample_size))
    
    stats = {"kept": 0, "rejected": 0}

    print(f"Analyse de {len(sample_files)} images par Homogénéité...")
    for img_name in tqdm(sample_files):
        img_path = os.path.join(input_dir, img_name)
        
        # Garder les MOINS intacts
        if "MOINS" in img_name.upper():
            shutil.copy(img_path, os.path.join(output_dir, img_name))
            stats["kept"] += 1
            continue

        # Ouvrir et convertir en niveau de gris (requis pour la GLCM)
        img = Image.open(img_path).convert('L')
        arr = np.array(img)
        
        # 1. Calcul de la Matrice de Co-occurrence (GLCM)
        # On regarde les pixels voisins à une distance de 1 pixel dans 4 directions (0, 45, 90, 135 degrés)
        glcm = graycomatrix(arr, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], 
                           levels=256, symmetric=True, normed=True)
        
        # 2. Extraction de la propriété 'Homogénéité' (Moyenne sur les 4 directions)
        homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]

        # 3. Logique de tri (Une image lisse/artificielle a une homogénéité élevée)
        if homogeneity > homo_thresh:
            # C'est probablement une route ou un toit (Lisse)
            shutil.copy(img_path, os.path.join(rejected_dir, img_name))
            stats["rejected"] += 1
        else:
            # C'est probablement une prairie (Rugueuse)
            shutil.copy(img_path, os.path.join(output_dir, img_name))
            stats["kept"] += 1

    print(f"\n--- BILAN DE L'HOMOGÉNÉITÉ ---")
    print(f"Images gardées (KEEP - Prairies) : {stats['kept']}")
    print(f"Images rejetées (REJECT - Maisons/Routes) : {stats['rejected']}")
    print(f"Les intrus sont dans : {rejected_dir}")

# --- EXECUTION ---
# Assure-toi d'avoir installé : pip install scikit-image numpy Pillow tqdm
source = r"C:\Users\clari\Documents\MA3\Design_Project\training_data_extracted\training_data"

# Homo_thresh: Un bon point de départ est entre 0.70 et 0.85. 
# Si tu mets 1.0, tu ne gardes que le blanc pur sans aucun grain.
filter_by_texture(source, "dataset_KEEP", "dataset_REJECTED", sample_size=1000, homo_thresh=0.7)

Analyse de 1000 images par Homogénéité...


100%|██████████| 1000/1000 [00:22<00:00, 43.56it/s]


--- BILAN DE L'HOMOGÉNÉITÉ ---
Images gardées (KEEP - Prairies) : 1000
Images rejetées (REJECT - Maisons/Routes) : 0
Les intrus sont dans : dataset_REJECTED


In [ ]:
# Checking the class balance
import re, glob

plus  = len(glob.glob(os.path.join(DATA_DIR, '*_Plus.tif')))
moins = len(glob.glob(os.path.join(DATA_DIR, '*_Moins.tif')))

print(f"Plus:  {plus}")
print(f"Moins: {moins}")
print(f"Ratio: {max(plus,moins)/min(plus,moins):.2f}x imbalance") #if ratio is very high (over 5) we may need to handle class imbalance in training

Plus:  59319
Moins: 31607
Ratio: 1.88x imbalance


In [7]:
import os
import re
import glob
from PIL import Image
from torch.utils.data import Dataset

class PrairieDataset(Dataset):

    # mapping between label class names and indices
    LABEL_CLASSES = {
        'Plus'  : 1,
        'Moins' : 0,
    }

    def __init__(self, data_dir, transform=None, split='train',
                 val_size=0.15, test_size=0.15):
        self.transform = transform
        self.data_dir  = data_dir

        # LOAD ALL PATCHES 
        all_patches = []
        for path in glob.glob(os.path.join(data_dir, '*.tif')):
            filename = os.path.basename(path)

            # parse filename: {x}_{y}_{qualite}.tif
            match = re.match(r'([\d.]+)_([\d.]+)_(.+)\.tif$', filename)
            if not match:
                continue

            x       = float(match.group(1))
            y       = float(match.group(2))
            qualite = match.group(3)
            
            if qualite not in self.LABEL_CLASSES:
                continue  # skips anything that's not Plus or Moins

            all_patches.append((path, x, y, self.LABEL_CLASSES[qualite]))

        # SPATIAL SPLIT by easting (x coordinate)
        all_patches.sort(key=lambda p: p[1])
        n         = len(all_patches)
        train_end = int(n * (1 - val_size - test_size))
        val_end   = int(n * (1 - test_size))

        if split == 'train':
            self.data = [(p[0], p[3]) for p in all_patches[:train_end]]
        elif split == 'val':
            self.data = [(p[0], p[3]) for p in all_patches[train_end:val_end]]
        elif split == 'test':
            self.data = [(p[0], p[3]) for p in all_patches[val_end:]]
        else:
            raise ValueError(f"Unknown split '{split}'. Use 'train', 'val', or 'test'.")

        print(f"Split '{split}': {len(self.data)} patches")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, x):
        img_path, label = self.data[x]

        img = Image.open(img_path).convert('RGB')

        if self.transform is not None:
            img = self.transform(img)

        return img, label

In [8]:
# Check if East Split ok

import re, glob, os
import numpy as np

DATA_DIR = os.path.expanduser('~/Documents/MA3/Design_Project/training_data_extracted/training_data')

patches = []
for path in glob.glob(os.path.join(DATA_DIR, '*.tif')):
    filename = os.path.basename(path)
    match = re.match(r'([\d.]+)_([\d.]+)_(.+)\.tif$', filename)
    if not match:
        continue
    x       = float(match.group(1))
    qualite = match.group(3)
    if qualite in ('Plus', 'Moins'):
        patches.append((x, qualite))

patches.sort(key=lambda p: p[0])
n         = len(patches)
train_end = int(n * 0.8)
val_end   = int(n * 0.9)

splits = {
    'train' : patches[:train_end],
    'val'   : patches[train_end:val_end],
    'test'  : patches[val_end:]
}

print(f"{'Split':<8} {'Total':>8} {'Plus':>8} {'Moins':>8} {'Plus%':>8}")
print("─" * 45)
for split_name, split_patches in splits.items():
    total = len(split_patches)
    plus  = sum(1 for p in split_patches if p[1] == 'Plus')
    moins = total - plus
    print(f"{split_name:<8} {total:>8} {plus:>8} {moins:>8} {100*plus/total:>7.1f}%")

Split       Total     Plus    Moins    Plus%
─────────────────────────────────────────────
train       72740    47044    25696    64.7%
val          9093     6396     2697    70.3%
test         9093     5879     3214    64.7%


In [48]:
# Checking the class balance
import re, glob

plus  = len(glob.glob(os.path.join(DATA_DIR, '*_Plus.tif')))
moins = len(glob.glob(os.path.join(DATA_DIR, '*_Moins.tif')))

print(f"Plus:  {plus}")
print(f"Moins: {moins}")
print(f"Ratio: {max(plus,moins)/min(plus,moins):.2f}x imbalance") #if ratio is very high (over 5) we may need to handle class imbalance in training

Plus:  59319
Moins: 31607
Ratio: 1.88x imbalance


In [2]:
import re, glob, os

DATA_DIR = os.path.expanduser('~/Documents/MA3/Design_Project/training_data_extracted/training_data')

# On stocke (coordonnée_x, chemin_complet, label_numérique)
patches = []
for path in glob.glob(os.path.join(DATA_DIR, '*.tif')):
    filename = os.path.basename(path)
    match = re.match(r'([\d.]+)_([\d.]+)_(.+)\.tif$', filename)
    if not match:
        continue
    
    x = float(match.group(1))
    qualite_str = match.group(3)
    
    # Conversion en chiffres pour PyTorch
    if qualite_str == 'Plus':
        label = 1
    elif qualite_str == 'Moins':
        label = 0
    else:
        continue
        
    # On garde le chemin (path) pour pouvoir ouvrir l'image plus tard
    patches.append({'x': x, 'path': path, 'label': label})

# Tri spatial Est-Ouest
patches.sort(key=lambda p: p['x'])

n = len(patches)
train_end = int(n * 0.8)
val_end   = int(n * 0.9)

# Dictionnaire de listes de dictionnaires
splits = {
    'train' : patches[:train_end],
    'val'   : patches[train_end:val_end],
    'test'  : patches[val_end:]
}

In [1]:
import torch
import numpy as np
import os
import re
import glob
from PIL import Image
from tqdm import tqdm
from torchvision import transforms

def get_train_split_stats(train_patches):
    """
    Calcule Mean et Std uniquement sur la liste des patches de train.
    train_patches: liste de dictionnaires [{'x':..., 'path':..., 'label':...}, ...]
    """
    sums = np.zeros(3)
    sq_sums = np.zeros(3)
    num_pixels = 0

    print(f"Calcul des stats sur le split TRAIN ({len(train_patches)} images)...")
    
    for item in tqdm(train_patches):
        path = item['path']
        try:
            with Image.open(path) as img_raw:
                # Conversion RGB et normalisation 0-1
                img = np.array(img_raw.convert('RGB')) / 255.0
                
                # Sommes pour la moyenne et l'écart-type
                sums += img.sum(axis=(0, 1))
                sq_sums += (img**2).sum(axis=(0, 1))
                num_pixels += img.shape[0] * img.shape[1]
        except Exception as e:
            print(f"Erreur sur l'image {path}: {e}")

    if num_pixels == 0:
        return None, None

    mean = sums / num_pixels
    std = np.sqrt(np.maximum((sq_sums / num_pixels) - mean**2, 1e-6))
    
    return mean, std

DATA_DIR = r"C:\Users\clari\Documents\MA3\Design_Project\training_data_extracted\training_data"
patches = []

for path in glob.glob(os.path.join(DATA_DIR, '*.tif')):
    filename = os.path.basename(path)
    match = re.match(r'([\d.]+)_([\d.]+)_(.+)\.tif$', filename)
    if match:
        x = float(match.group(1))
        qualite_str = match.group(3)
        label = 1 if qualite_str == 'Plus' else 0
        patches.append({'x': x, 'path': path, 'label': label})

# Tri spatial Est-Ouest
patches.sort(key=lambda p: p['x'])
train_end = int(len(patches) * 0.8)
train_split = patches[:train_end]

# --- 2. CALCUL DES VALEURS ---
final_mean, final_std = get_train_split_stats(train_split)

if final_mean is not None:
    print(f"\n--- RÉSULTATS À COPIER ---")
    print(f"Mean: {final_mean.tolist()}")
    print(f"Std:  {final_std.tolist()}")

    # --- 3. PIPELINE DE TRANSFORMATION ---
    # Ces valeurs seront utilisées pour le train ET la validation
    transform_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(), 
        transforms.Normalize(mean=final_mean.tolist(), std=final_std.tolist())
    ])

Calcul des stats sur le split TRAIN (72740 images)...


100%|██████████| 72740/72740 [17:56<00:00, 67.59it/s]


--- RÉSULTATS À COPIER ---
Mean: [0.3973067899556195, 0.44638321989232976, 0.33463927838161495]
Std:  [0.15175289831701147, 0.14110251706340815, 0.12873382339540532]


In [ ]:
#changer ici avec les vraies valeurs de mean std etc...

my_mean = [0.3949478 , 0.44614649 , 0.33258339]
my_std =  [0.15113239 , 0.14074013 , 0.12560346]#A remplacer ici


train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),      # Obligatoire pour ResNet-50
    transforms.RandomHorizontalFlip(p=0.5), # Retourne l'image 1 fois sur 2
    transforms.RandomVerticalFlip(p=0.5),   
    transforms.RandomRotation(90),          # Rotation aléatoire
    transforms.ToTensor(),                  # Convertit en [0, 1]
    transforms.Normalize(mean=my_mean, std=my_std) # Normalise
])


In [ ]:
'''

import os
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

def get_stats_and_clean(image_dir, output_dir, vari_threshold=0.05, sample_size=10000):
    """
    Combine le nettoyage, le calcul des indices et les stats de normalisation.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    files = [f for f in os.listdir(image_dir) if f.endswith('.tif')]
    cleaned_data = []
    
    print(f"Filtrage et Extraction des labels ---")
    for img_name in tqdm(files):
        img_path = os.path.join(image_dir, img_name)
        img = Image.open(img_path).convert('RGB')
        arr = np.array(img).astype(float)
        
        # Calcul VARI pour filtrage
        R, G, B = arr[:,:,0], arr[:,:,1], arr[:,:,2]
        vari = (G - R) / (G + R - B + 1e-6)
        mean_vari = np.mean(vari)

        if mean_vari > vari_threshold:
            # On garde le patch
            shutil.copy(img_path, os.path.join(output_dir, img_name))
            
            # Extraction du label (on cherche 'Q2' dans le nom)
            label = 1 if "Q2" in img_name.upper() else 0
            
            cleaned_data.append({
                'filename': img_name, 
                'label': label, 
                'vari_mean': mean_vari,
                'vari_std': np.std(vari),
                'vari_q50': np.median(vari)
            })

    df = pd.DataFrame(cleaned_data)
    df.to_csv(os.path.join(output_dir, "metadata_cleaned.csv"), index=False)
    
    print(f"\nNettoyage terminé : {len(df)} images conservées sur {len(files)}.")
    print(f"Répartition des classes :\n{df['label'].value_counts(normalize=True) * 100}")

    print(f"\n Calcul des stats de normalisation (RGB) ---")
    # On calcule les stats uniquement sur le dossier propre
    clean_files = [f for f in os.listdir(output_dir) if f.endswith('.tif')]
    if len(clean_files) > sample_size:
        subset = np.random.choice(clean_files, sample_size, replace=False)
    else:
        subset = clean_files

    pixel_values = []
    for img_name in tqdm(subset):
        img = Image.open(os.path.join(output_dir, img_name)).convert('RGB')
        img_array = np.array(img) / 255.0  # On travaille en float 0-1
        pixel_values.append(img_array.reshape(-1, 3))
    
    pixel_values = np.concatenate(pixel_values, axis=0)
    mean = pixel_values.mean(axis=0)
    std = pixel_values.std(axis=0)
    
    return mean, std, df

# --- EXÉCUTION ---

PATH_TRAIN_RAW = "small_sample" 
PATH_TRAIN_CLEANED = "train_final_small_sample"

final_mean, final_std, df_meta = get_stats_and_clean(PATH_TRAIN_RAW, PATH_TRAIN_CLEANED)

print(f"\n--- RÉSULTATS POUR TON CODE PYTORCH ---")
print(f"Moyenne (Mean) : {final_mean}")
print(f"Écart-type (Std) : {final_std}")


#ensuite on peut utiliser final_mean et final_std pour la normalisation dans ton code PyTorch avec le code suivant 
# normalize = transforms.Normalize(mean=final_mean, std=final_std)

'''